# Open-Weight VLM Recipes: What Actually Matters: Interactive Visual Explorer

> The 2024-2026 open-weight VLM literature is a forest of ablation tables. Apple's MM1 tested 13 combinations of image encoder, connector, and data mix. Allen AI's Molmo proved detailed human captions beat GPT-4V distillation. Cambrian-1 ran 20+ encoder comparisons. Idefics2 formalized the five-axis design space. Prismatic VLMs compared 27 training recipes on a controlled benchmark. Out of all that noise, a small set of results holds across papers: image encoder matters more than connector architecture, data mixture matters more than either, and detailed human captions beat distilled synthetic data. This lesson reads those tables so you do not have to.

Welcome to the interactive companion notebook for **Open-Weight VLM Recipes: What Actually Matters**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Open-weight VLM recipe picker — condensed ablation tables from 2024-2025 papers.

Encodes the key findings from MM1, Idefics2, Cambrian-1, Molmo, Prismatic VLMs
as simple data tables. Lets you ask:
  - given a budget and task mix, which recipe wins
  - if I swap axis X, what is the expected delta
  - which axis to ablate first

No numpy, no pandas — just dicts and print tables. The point is the structure
of the evidence, not the numeric precision.
"""


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

@dataclass
class Recipe:
    name: str
    encoder: str
    connector: str
    llm_b: int
    data: str
    resolution: str
    mmmu: float
    cv_bench: float
    docvqa: float

RECIPES = [
    Recipe("LLaVA-1.5", "CLIP L/14 @336", "MLP-2", 13, "LLaVA-Inst-150k", "336", 35.3, 56.0, 55.0),
    Recipe("LLaVA-NeXT", "CLIP L/14 @336", "MLP-2", 13, "LLaVA-Inst + shareGPT4V", "AnyRes 672", 36.2, 58.5, 77.4),
    Recipe("Idefics2-8B", "SigLIP SO400m/14", "Perceiver-64", 7, "OBELICS + Cauldron", "980 split", 43.0, 60.0, 74.0),
    Recipe("MM1-3B", "CLIP L/14", "C-Abstractor", 3, "interleaved + caption", "672", 38.6, 59.0, 62.0),
    Recipe("MM1-30B", "CLIP L/14", "C-Abstractor", 30, "interleaved + caption", "672", 44.7, 64.0, 74.0),
    Recipe("Molmo-7B-D", "SigLIP SO400m/14", "MLP-2", 7, "PixMo (712K human caps)", "AnyRes 672", 45.3, 65.0, 92.4),
    Recipe("Molmo-72B", "SigLIP SO400m/14", "MLP-2", 72, "PixMo (712K human caps)", "AnyRes 672", 54.1, 73.0, 93.5),
    Recipe("Cambrian-1-8B", "CLIP + DINOv2 + SigLIP + ConvNeXt", "SVA", 8, "Cambrian-10M", "672", 42.7, 67.8, 77.8),
    Recipe("Prismatic-7B default", "SigLIP SO400m/14", "MLP-2", 7, "LLaVA-Inst + shareGPT4V", "336", 40.0, 58.0, 70.0),
]


In [ ]:
def axis_impact() -> None:
    print("\nAXIS-IMPACT DECOMPOSITION (Prismatic VLMs controlled comparison)")
    print("-" * 60)
    axes = [
        ("visual-token count", 60, "64 -> 576 -> 1024 tokens; diminishing past 1024"),
        ("image encoder",      20, "CLIP vs SigLIP vs DINOv2; concatenation helps"),
        ("connector arch",      5, "MLP ~= Q-Former ~= Perceiver at same token count"),
        ("data mix",           10, "detailed human caps > distilled GPT-4V data"),
        ("LLM size",           15, "7B -> 70B plateau around MMMU 55"),
        ("resolution sched",    5, "ramp 224 -> 448 > flat 448; native wins OCR"),
    ]
    total_weight = sum(a[1] for a in axes)
    print(f"{'axis':<22}{'%var':>8}  note")
    for name, pct, note in axes:
        bar = "#" * (pct // 2)
        print(f"{name:<22}{pct:>6}% {bar}")
        print(f"{'':<22}       {note}")
    print(f"note: weights rebased from ~{total_weight}% to ~100% after rounding.")


In [ ]:
def compare_encoders() -> None:
    print("\nENCODER SWAP DELTAS (fixed 7B LLM, LLaVA-Inst + shareGPT4V data)")
    print("-" * 60)
    rows = [
        ("CLIP ViT-L/14 @ 336",        38.5, 56.0, 70.0),
        ("SigLIP SO400m/14 @ 384",     41.0, 60.0, 75.0),
        ("DINOv2 ViT-g/14 @ 224",      37.0, 65.0, 52.0),
        ("SigLIP + DINOv2 concat",     42.0, 67.0, 74.0),
        ("InternViT-6B @ 448",         43.0, 66.0, 78.0),
    ]
    print(f"{'encoder':<32}{'MMMU':>8}{'CV-B':>8}{'DocVQA':>10}")
    for name, mmmu, cv, doc in rows:
        print(f"{name:<32}{mmmu:>8.1f}{cv:>8.1f}{doc:>10.1f}")
    print("deltas: SigLIP adds +2.5 MMMU over CLIP; DINOv2 wins CV-Bench; "
          "concat beats either alone on vision-centric bench.")


In [ ]:
def compare_data() -> None:
    print("\nDATA-MIX DELTAS (fixed SigLIP + 7B LLM + AnyRes)")
    print("-" * 60)
    rows = [
        ("LLaVA-Inst-150k",         40.0, "web caps + GPT-4 dialogues"),
        ("+ ShareGPT4V",            42.0, "+ GPT-4V detailed captions"),
        ("+ Cauldron",              43.0, "+ OCR + charts + multimodal instructions"),
        ("PixMo (human caps only)", 45.3, "712K dense human captions"),
        ("PixMo + Cauldron + more", 47.0, "best data mix as of Jul 2025"),
    ]
    print(f"{'data mix':<28}{'MMMU':>8}  notes")
    for name, mmmu, note in rows:
        print(f"{name:<28}{mmmu:>8.1f}  {note}")
    print("finding: dense human captions beat distilled captions by +2-3 MMMU")
    print("         at the same training token count (Molmo thesis).")


In [ ]:
def print_recipes() -> None:
    print("\nCANONICAL OPEN VLMS (ablation-reported MMMU, CV-Bench, DocVQA)")
    print("-" * 60)
    print(f"{'recipe':<22}{'LLM':>6}{'MMMU':>8}{'CV-B':>8}{'DocVQA':>10}")
    for r in RECIPES:
        print(f"{r.name:<22}{r.llm_b:>5}B{r.mmmu:>8.1f}{r.cv_bench:>8.1f}{r.docvqa:>10.1f}")

def pick_recipe(budget_b: int, task: str) -> None:
    print(f"\nPICKER: budget {budget_b}B params, task profile: {task}")
    print("-" * 60)
    weights = {"mmmu": 1.0, "cv": 1.0, "doc": 1.0}
    if task == "ocr":
        weights = {"mmmu": 0.4, "cv": 0.3, "doc": 1.2}
    elif task == "agent":
        weights = {"mmmu": 1.0, "cv": 1.2, "doc": 0.8}
    elif task == "reasoning":
        weights = {"mmmu": 1.5, "cv": 0.5, "doc": 0.8}


In [ ]:
def score(r: Recipe) -> float:
        return r.mmmu * weights["mmmu"] + r.cv_bench * weights["cv"] + r.docvqa * weights["doc"]

candidates = [r for r in RECIPES if r.llm_b <= budget_b]
    candidates.sort(key=score, reverse=True)
    for r in candidates[:3]:
        print(f"  {r.name:<22} LLM {r.llm_b}B  score={score(r):.1f}")
        print(f"    encoder={r.encoder}")
        print(f"    data   ={r.data}")
        print(f"    res    ={r.resolution}")


In [ ]:
def main() -> None:
    print("=" * 60)
    print("OPEN-WEIGHT VLM RECIPE PICKER (Phase 12, Lesson 07)")
    print("=" * 60)

print_recipes()
    axis_impact()
    compare_encoders()
    compare_data()

pick_recipe(10, "ocr")
    pick_recipe(80, "reasoning")
    pick_recipe(10, "agent")

if __name__ == "__main__":
    main()
